# Temporal filters from naturalistic stimuli

LPD on the delay-embedded scalar dead-leaves signal (Section 4, Fig. 3b right).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

RESULTS_DIR = "results"  # output folder for figures
DATA_DIR = "data"  # output folder for intermediate results
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Stimulus, increments and LPD

In [ ]:
T = 10000
t = np.arange(1, T + 1)

# 1D dead-leaves signal: piecewise-constant plateaus with smooth edges
plateau_vals = np.array(
    [
        0, 1, -0.5, 1.5, 0.3, 1.1, -0.7, 1.2, -0.2, 0.1, 1.3, -0.75, 1.25, 0.33, 1.21,
        -0.78, 1.22, -0.125, 0.04, 1.16, -0.52, 1.55, 0.36, -1.15, -0.73, 1.29, -0.22,
        0.17, -1.3, -0.55, -1.25, 0.43, 1.11, -0.98, 1.75, -0.9
    ]
)
num_transitions = plateau_vals.size - 1

transition_centers = np.linspace(T / num_transitions, T, num_transitions + 1)
transition_centers = transition_centers[1:]
transition_width = 5.0

x = plateau_vals[0] * np.ones(T)
for k in range(num_transitions):
    v_left = plateau_vals[k]
    v_right = plateau_vals[k + 1]
    c_k = transition_centers[k]

    s_k = 0.5 * (1 + np.tanh((t - c_k) / transition_width))
    x = x + (v_right - v_left) * s_k

nl = 1e-4
a = x + nl * np.random.randn(T)

margin = 100
length = 10
X = []
Y = []
# Past/future increments of the delay-embedded signal
for start_y in range(margin, a.shape[0] - margin):

    x = (
        a[start_y - length + 1 : start_y + length]
        - a[start_y - length : start_y + length - 1]
    )
    y = (
        a[start_y - length + 2 * length + 1 : start_y + length + 2 * length]
        - a[start_y - length + 2 * length : start_y + length + 2 * length - 1]
    )
    x += 1e-6 * np.random.randn(x.shape[0])
    y += 1e-6 * np.random.randn(y.shape[0])

    X.append(x)
    Y.append(y)

X = np.array(X)
Y = np.array(Y)

X = torch.from_numpy(X).to(device)
Y = torch.from_numpy(Y).to(device)

Cxx = X.T @ X / T / 2
Cxx += 2e-5 * torch.eye(Cxx.shape[0]).to(device)
Cxy = X.T @ Y / T / 2
Cyy = Y.T @ Y / T / 2
Cyy += 2e-5 * torch.eye(Cyy.shape[0]).to(device)

Nz_max = 100
toll = 1e-8
a_z = 0.1
a_W = 0.1
a_M = 0.1

T, n = X.shape
m = 4
rf = torch.randn(m, n).to(device)
error = 10.0
count = 0


def orthogonal_matrix(dim):
    Q, _ = np.linalg.qr(np.random.randn(dim, dim))
    return Q

M = np.eye(m)
W = orthogonal_matrix(m) @ np.eye(m, n) @ orthogonal_matrix(n) * 0.1
M = torch.from_numpy(M).to(device)
W = torch.from_numpy(W).to(device)
Z = torch.rand(T, m).to(device).double()
Z[Z < 0.0] = 0.0

# Offline LPD with rectified inference
with torch.no_grad():
    while count <= 8000:

        for _ in range(Nz_max):
            Z_new = (W @ X.T - M @ Z.T).T
            Z = a_z * Z_new + Z
            Z[Z < 0.0] = 0.0
            if (
                torch.linalg.norm(Z_new - Z, ord="fro")
                / (torch.linalg.norm(Z, ord="fro") + 1e-12)
                < toll
            ):
                break
        Z = Z / torch.linalg.norm(Z, axis=0, keepdims=True)

        M_new = Z.T @ Z
        Czx = Z.T @ X / np.sqrt(T)
        W_new = (
            Czx
            @ torch.linalg.inv(Cxx)
            @ Cxy
            @ torch.linalg.inv(Cyy)
            @ Cxy.T
            @ torch.linalg.inv(Cxx)
        )
        M = a_M * M_new + (1 - a_M) * M
        W = a_W * W_new + (1 - a_W) * W

        new_rf = torch.linalg.inv(M + 1e-6 * torch.eye(M.shape[0]).to(device)) @ W
        new_rf = new_rf / torch.linalg.norm(new_rf, axis=1, keepdims=True)
        error = torch.abs(rf - new_rf).sum() / torch.abs(rf).sum()
        rf = new_rf

        count += 1
        if count % 300 == 1:
            print(error)
        if error < 1e-4:
            break

In [ ]:
plt.figure(figsize=(6, 4))
colors = ["orange", "blue", "red", "green"]
for i in range(m):
    plt.plot(
        rf.detach().cpu()[i], color=colors[i], label="Filter {}".format(i + 1), lw=3
    )

plt.xlabel("Time-lag (pixel)", fontsize=15)

plt.xlim(0, 5)

plt.xticks([18, 10, 2], ["0", "8", "16"], fontsize=13)
plt.yticks([-1, 0, 1], fontsize=13)
plt.ylabel("Temporal filter", fontsize=15)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.invert_xaxis()

plt.savefig(
    os.path.join(RESULTS_DIR, "temporal_filters.png"), dpi=600, bbox_inches="tight"
)